# Olist - Data Preprocessing (ETL to Snowflake Schema) - Kaggle version

This notebook transforms raw data (read directly from the **SQLite** file `olist.sqlite`) into Fact/Dimension tables matching the `olist_snowflake_schema.drawio` design.

**How to use on Kaggle:** go to **Add Input** → select the dataset containing `olist.sqlite` (or `.db`) → set `SQLITE_PATH` below to the real path → run the notebook.

**Fact tables:**
- `FACT_ORDER_ITEMS` - grain: 1 row / order item
- `FACT_ORDER_PAYMENT` - grain: 1 row / payment (order_id + payment_sequential)
- `FACT_SELLER_LEAD` - grain: 1 row / lead (merged leads_qualified + leads_closed)

**Dimension tables:**
- `DIM_CUSTOMER`, `DIM_SELLER`, `DIM_PRODUCT`, `DIM_PRODUCT_CATEGORY`, `DIM_LOCATION`, `DIM_REVIEW`

Data quality issues handled (found during analysis):
1. `geolocation_zip_code_prefix` is **not unique** (~52 rows/zip) → `DIM_LOCATION` keeps every original row (no aggregation); FKs are matched on `(zip, city, state)` together to reduce ambiguity.
2. `product_category_name` may not exist in `product_category_name_translation` → missing categories are appended instead of using an INNER JOIN (which would silently drop rows).
3. `order_payments` and `order_reviews` have N rows per 1 order (fan-out) → kept as separate tables/fact, not merged directly into `FACT_ORDER_ITEMS`.
4. Zip columns in SQLite are stored as **INTEGER** → leading zeros are lost on read → always re-cast to a zero-padded string (zfill) after loading.
5. `leads_closed.seller_id` has ~55% of rows not present in `sellers` (new sellers who haven't sold anything yet) → the FK is still kept, orphans are accepted, and flagged with an `is_won` column.


## 1. Import libraries

In [ ]:
import sqlite3
import glob
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)


## 2. Connect to the SQLite file on Kaggle

Kaggle loads datasets via **Add Input**, and the data lives under `/kaggle/input/<dataset-name>/...`.
**Edit `SQLITE_PATH` below to the real path** of the `.sqlite` file after adding the input
(check the Input panel on the right, or run `!find /kaggle/input -name "*.sqlite"` in a separate cell to get the exact path).

In [ ]:
SQLITE_PATH = Path('/kaggle/input/your-dataset/olist.sqlite')   # <-- change to the real path on Kaggle

IN_KAGGLE = Path('/kaggle/input').exists()
OUT_DIR = Path('/kaggle/working/processed') if IN_KAGGLE else Path('processed')
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert SQLITE_PATH.exists(), f'File not found: {SQLITE_PATH} - check the path after Add Input'

conn = sqlite3.connect(SQLITE_PATH)
print('SQLITE_PATH =', SQLITE_PATH.resolve())
print('OUT_DIR     =', OUT_DIR.resolve())
print('Tables found in sqlite:')
print(pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn))


## 3. Load raw data (query directly from SQLite)

In [ ]:
customers   = pd.read_sql_query('SELECT * FROM customers', conn)
geolocation = pd.read_sql_query('SELECT * FROM geolocation', conn)
leads_closed    = pd.read_sql_query('SELECT * FROM leads_closed', conn)
leads_qualified = pd.read_sql_query('SELECT * FROM leads_qualified', conn)
order_items    = pd.read_sql_query('SELECT * FROM order_items', conn)
order_payments = pd.read_sql_query('SELECT * FROM order_payments', conn)
order_reviews  = pd.read_sql_query('SELECT * FROM order_reviews', conn)
orders         = pd.read_sql_query('SELECT * FROM orders', conn)
category_translation = pd.read_sql_query('SELECT * FROM product_category_name_translation', conn)
products = pd.read_sql_query('SELECT * FROM products', conn)
sellers  = pd.read_sql_query('SELECT * FROM sellers', conn)

conn.close()

tables = {
    'customers': customers, 'geolocation': geolocation, 'leads_closed': leads_closed,
    'leads_qualified': leads_qualified, 'order_items': order_items, 'order_payments': order_payments,
    'order_reviews': order_reviews, 'orders': orders, 'category_translation': category_translation,
    'products': products, 'sellers': sellers,
}
for name, df in tables.items():
    print(f'{name:22s} shape={df.shape}')


## 4. Data quality checks

Re-confirm the issues found before processing, so the notebook proves for itself why the transformation steps below are needed.

In [ ]:
# 4.1 zip code in geolocation is NOT unique
dup_zip = geolocation['geolocation_zip_code_prefix'].value_counts()
print('Distinct zip codes in geolocation:', dup_zip.shape[0])
print('Total rows in geolocation:', len(geolocation))
print('Average rows per zip:', round(len(geolocation) / dup_zip.shape[0], 2))
assert dup_zip.shape[0] < len(geolocation), 'geolocation_zip_code_prefix is not unique -> must handle this when building the FK'


In [ ]:
# 4.2 product_category_name may be missing from the translation table
missing_categories = set(products['product_category_name'].dropna().unique()) - \
                     set(category_translation['product_category_name'])
print('Categories in products but NOT in category_translation:', len(missing_categories))
print(sorted(missing_categories)[:10])


In [ ]:
# 4.3 order_payments and order_reviews have N rows per 1 order (fan-out)
pay_per_order = order_payments.groupby('order_id').size()
rev_per_order = order_reviews.groupby('order_id').size()
print('order_payments: max rows / order =', pay_per_order.max(), '| orders with >1 payment row =', (pay_per_order > 1).sum())
print('order_reviews : max rows / order =', rev_per_order.max(), '| orders with >1 review row  =', (rev_per_order > 1).sum())


In [ ]:
# 4.4 leads_closed.seller_id orphans against sellers
orphan_sellers = set(leads_closed['seller_id']) - set(sellers['seller_id'])
print(f'leads_closed has {len(orphan_sellers)}/{leads_closed["seller_id"].nunique()} seller_id NOT present in sellers'
      f' ({len(orphan_sellers)/leads_closed["seller_id"].nunique():.1%})')


## 5. Normalize zip codes (keep leading zeros)

SQLite stores the zip columns as **INTEGER**, so leading zeros (e.g. `01037`) are already lost when read (`1037`).
Cast every zip column to a 5-character zero-padded string so matching between tables is correct.

In [ ]:
def normalize_zip(series):
    return series.astype(str).str.extract(r'(\d+)')[0].str.zfill(5)

customers['customer_zip_code_prefix'] = normalize_zip(customers['customer_zip_code_prefix'])
sellers['seller_zip_code_prefix'] = normalize_zip(sellers['seller_zip_code_prefix'])
geolocation['geolocation_zip_code_prefix'] = normalize_zip(geolocation['geolocation_zip_code_prefix'])


## 6. DIM_LOCATION

**No aggregation** - keep every original row of `geolocation` (each row is one sampled GPS point),
just add `location_id` as a sequential surrogate primary key.

Since zip_code_prefix is not unique, `DIM_CUSTOMER`/`DIM_SELLER` below will match on all 3 columns
`(zip_code_prefix, city, state)` to find the right `location_id` - instead of matching on zip alone.

In [ ]:
dim_location = geolocation.copy()
dim_location.insert(0, 'location_id', range(1, len(dim_location) + 1))

print('DIM_LOCATION:', dim_location.shape)
dim_location.head()


## 7. DIM_PRODUCT_CATEGORY

Append categories that exist in `products` but are missing an English translation (avoids losing data with an INNER JOIN).

In [ ]:
dim_product_category = category_translation.copy()

if missing_categories:
    extra = pd.DataFrame({
        'product_category_name': sorted(missing_categories),
        'product_category_name_english': np.nan,
    })
    dim_product_category = pd.concat([dim_product_category, extra], ignore_index=True)

print('DIM_PRODUCT_CATEGORY:', dim_product_category.shape)
dim_product_category.tail()


## 8. DIM_PRODUCT

In [ ]:
dim_product = products[[
    'product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght',
    'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm',
]].copy()

print('DIM_PRODUCT:', dim_product.shape)
dim_product.head()


## 9. DIM_CUSTOMER

Look up `location_id` by matching `(zip_code_prefix, city, state)` - case-insensitive -
against `DIM_LOCATION`. Since the same triple can still have several duplicate GPS points, take the `location_id` of the **first** match.

In [ ]:
geo_lookup = (
    dim_location
    .assign(
        _city_lc=dim_location['geolocation_city'].str.lower(),
        _state_uc=dim_location['geolocation_state'].str.upper(),
    )
    .drop_duplicates(subset=['geolocation_zip_code_prefix', '_city_lc', '_state_uc'], keep='first')
    [['location_id', 'geolocation_zip_code_prefix', '_city_lc', '_state_uc']]
)

customers['_city_lc'] = customers['customer_city'].str.lower()
customers['_state_uc'] = customers['customer_state'].str.upper()

dim_customer = customers.merge(
    geo_lookup,
    left_on=['customer_zip_code_prefix', '_city_lc', '_state_uc'],
    right_on=['geolocation_zip_code_prefix', '_city_lc', '_state_uc'],
    how='left',
)[['customer_id', 'customer_unique_id', 'location_id']]

missing_loc = dim_customer['location_id'].isna().sum()
print('DIM_CUSTOMER:', dim_customer.shape, '| rows with no location_id found:', missing_loc)
dim_customer.head()


## 10. DIM_SELLER

Same `(zip_code_prefix, city, state)` matching approach as DIM_CUSTOMER.

In [ ]:
sellers['_city_lc'] = sellers['seller_city'].str.lower()
sellers['_state_uc'] = sellers['seller_state'].str.upper()

dim_seller = sellers.merge(
    geo_lookup,
    left_on=['seller_zip_code_prefix', '_city_lc', '_state_uc'],
    right_on=['geolocation_zip_code_prefix', '_city_lc', '_state_uc'],
    how='left',
)[['seller_id', 'location_id']]

missing_loc = dim_seller['location_id'].isna().sum()
print('DIM_SELLER:', dim_seller.shape, '| rows with no location_id found:', missing_loc)
dim_seller.head()


## 11. DIM_REVIEW

Drop duplicates on `review_id` (if any) and parse the date columns.

In [ ]:
dim_review = order_reviews.drop_duplicates(subset='review_id').copy()
dim_review['review_creation_date'] = pd.to_datetime(dim_review['review_creation_date'], errors='coerce')
dim_review['review_answer_timestamp'] = pd.to_datetime(dim_review['review_answer_timestamp'], errors='coerce')

dim_review = dim_review[[
    'review_id', 'order_id', 'review_score', 'review_comment_title',
    'review_comment_message', 'review_creation_date', 'review_answer_timestamp',
]]

print('DIM_REVIEW:', dim_review.shape)
dim_review.head()


## 12. FACT_ORDER_ITEMS

Grain: 1 row / order item. Pulls `customer_id` from `orders`.

In [ ]:
fact_order_items = order_items.merge(
    orders[['order_id', 'customer_id']], on='order_id', how='left',
)[['order_id', 'order_item_id', 'customer_id', 'seller_id', 'product_id', 'price', 'freight_value']]

print('FACT_ORDER_ITEMS:', fact_order_items.shape)
fact_order_items.head()


## 13. FACT_ORDER_PAYMENT

Grain: 1 row / payment (`order_id` + `payment_sequential`) - kept as its own table, not merged into FACT_ORDER_ITEMS because it has a different grain.

In [ ]:
fact_order_payment = order_payments[[
    'order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value',
]].copy()

print('FACT_ORDER_PAYMENT:', fact_order_payment.shape)
fact_order_payment.head()


## 14. FACT_SELLER_LEAD

Grain: 1 row / lead. `LEFT JOIN leads_qualified <- leads_closed` on `mql_id` (1-0..1 relationship, confirmed no duplicates and no orphans on the leads_closed side).
Adds an `is_won` flag marking leads that converted into a seller.

In [ ]:
fact_seller_lead = leads_qualified.merge(leads_closed, on='mql_id', how='left')
fact_seller_lead['is_won'] = fact_seller_lead['seller_id'].notna()

print('FACT_SELLER_LEAD:', fact_seller_lead.shape)
print('Number of won leads:', fact_seller_lead['is_won'].sum(), '/', len(fact_seller_lead))
fact_seller_lead.head()


## 15. Referential integrity checks

Confirm that the Fact tables just built have FKs matching their corresponding Dimensions (except for the cases already known to have orphans).

In [ ]:
def check_fk(fact_df, fact_col, dim_df, dim_col, label):
    orphan = set(fact_df[fact_col].dropna()) - set(dim_df[dim_col].dropna())
    print(f'{label:45s} orphan = {len(orphan):6d} / {fact_df[fact_col].nunique()}')
    return orphan

check_fk(fact_order_items, 'customer_id', dim_customer, 'customer_id', 'FACT_ORDER_ITEMS.customer_id -> DIM_CUSTOMER')
check_fk(fact_order_items, 'seller_id',   dim_seller,   'seller_id',   'FACT_ORDER_ITEMS.seller_id -> DIM_SELLER')
check_fk(fact_order_items, 'product_id',  dim_product,  'product_id',  'FACT_ORDER_ITEMS.product_id -> DIM_PRODUCT')
check_fk(dim_product,  'product_category_name', dim_product_category, 'product_category_name', 'DIM_PRODUCT.product_category_name -> DIM_PRODUCT_CATEGORY')
check_fk(dim_customer, 'location_id', dim_location, 'location_id', 'DIM_CUSTOMER.location_id -> DIM_LOCATION')
check_fk(dim_seller,   'location_id', dim_location, 'location_id', 'DIM_SELLER.location_id -> DIM_LOCATION')
check_fk(fact_order_payment, 'order_id', fact_order_items, 'order_id', 'FACT_ORDER_PAYMENT.order_id -> FACT_ORDER_ITEMS.order_id')
check_fk(dim_review,   'order_id', fact_order_items, 'order_id', 'DIM_REVIEW.order_id -> FACT_ORDER_ITEMS.order_id')
_ = check_fk(fact_seller_lead, 'seller_id', dim_seller, 'seller_id', 'FACT_SELLER_LEAD.seller_id -> DIM_SELLER (orphans expected)')


**Explaining the 2 unexpected orphans (not an ETL bug):**

- `FACT_ORDER_PAYMENT.order_id` has ~775 rows that don't match `FACT_ORDER_ITEMS`, and `DIM_REVIEW.order_id` has ~725 similar rows.
- Reason: `order_items` is only recorded once a product is **actually processed/shipped**, while a customer can still have **paid or left a review** for an order that was later `canceled`/`unavailable`, before any item row was ever created. Checking by `order_status` confirms >95% of these orphans fall into those two statuses - this is genuine business behavior, not a join bug.

## 16. Export results

Exported to `/kaggle/working/processed/` (Kaggle's **Output** tab will automatically show these files for download or for use as Input in another notebook).

In [ ]:
outputs = {
    'dim_customer.csv': dim_customer,
    'dim_seller.csv': dim_seller,
    'dim_product.csv': dim_product,
    'dim_product_category.csv': dim_product_category,
    'dim_location.csv': dim_location,
    'dim_review.csv': dim_review,
    'fact_order_items.csv': fact_order_items,
    'fact_order_payment.csv': fact_order_payment,
    'fact_seller_lead.csv': fact_seller_lead,
}

for filename, df in outputs.items():
    df.to_csv(OUT_DIR / filename, index=False)
    print(f'Exported {filename:28s} shape={df.shape}')
